We'll learn about **coroutines**, and why are they returned by async functions instead of normal values like regular python functions do, and give an idea of when to use the **async / await** keywords in an async function.

### Coroutine

It is an object that is returned by an async function, if you don't await it explicitly. An async function is the same as a python function with added functionality of *pausing* its execution with the help of some dedicated syntax (async / await).

As such, they behave quite differently from regular python functions, even if they look very similar in terms of code.

you use 'async' to define async functions.
you use 'await' to tell async functions when to pause.

in terms of terminology, you'd call a function with 'async def' written an 'async function' or 'coroutine function', which returns a 'coroutine object'.

`async function → called → produces a coroutine → awaited → produces the result`

In [ ]:
# Async Function vs Normal Python Functions

async def coroutine_add_one(number: int) -> int:
    x = 1
    y = 2
    return number + 1

def add_one(number: int) -> int:
    return number + 1

function_result = add_one(1)
coroutine_result = coroutine_add_one(1)

# no problem if you don't understand this rn
print(f"coroutine code : {coroutine_result.cr_code}")   # the code object (bytecode lives here)
frame = coroutine_result.cr_frame
if frame is not None:
    print(f"coroutine frame locals: {frame.f_locals}")  # local variables in the frame
    print(f"coroutine frame instruction pointer: {frame.f_lasti}")  # last instruction executed
else:
    print("coroutine frame: None")

print(f'Function result is {function_result} and the type is {type(function_result)}')
print(f'Coroutine result is {coroutine_result} and the type is {type(coroutine_result)}')

Notice how the async function doesn't give answer, it gives a coroutine object... This coroutine object is then 'awaited' to fetch us the result of the code...

Coroutine objects can only be run in event loops.

In [ ]:
import asyncio

async def coroutine_add_one(number: int) -> int:
    return number + 1

result = asyncio.run(coroutine_add_one(1)) # cute syntax for starting an event loop.
print(result) 

# NOTE: You will very likely use this asyncio.run() call in every program of yours which uses asyncio, and you'll use it exactly once... It's supposed to be the gateway into the event loop... As such, this call must execute a 'main' async fn, which sets into motion all of your other coroutines in the application.

### NOTE

in python community, 'coroutine' can refer to the 'coroutine object' or the 'coroutine function' that returned the object, depending on context. The line is thoda blurry.

### pausing execution with the 'await' keyword

Very important to understand that using the await keyword in an async function does not CHANGE the order of execution of statements...

for e.g.

```python
some_code()
await something()
more_code_dependent_on_something()
```

Here, the containing coroutine (the one which has the `await something()` call) DOES NOT move past `await something()`, until `something()`'s results have been returned. Don't make the mistake of thinking that execution moves to `more_code_dependent_on_something()` while `something()` has yet to return its value... Order is still maintained. What asyncio only does is fill up that empty time where we're waiting for `something()` to return, with another, independent task, if it's waiting in the queue... 

In [ ]:
import asyncio

async def add_one(number: int) -> int:
    return number + 1

async def main() -> None:
    one_plus_one = await add_one(1)
    two_plus_one = await add_one(2)
    print(one_plus_one)
    print(two_plus_one)

asyncio.run(main())

# NOTE: note that since add_one() does not await anything itself, this entire blurb of code will run pretty much like regular sequential code... To truly appreciate the concurrency in waiting that asyncio brings with itself, we'll have to simulate long-running operations by using the asyncio.sleep() command.

In [ ]:
import asyncio

async def hello_world_message() -> str:
    await asyncio.sleep(1) # here's how to use it
    return 'Hello World!'

async def main() -> None:
    message = await hello_world_message()
    print(message)

asyncio.run(main())

In [ ]:
# let's create a verbose 'delay' function which will have the coroutine sleep for however long we specify, printing when sleep starts and ends. Better for us to visualize the concurrency.

# (Added this in a module of its own)

In [ ]:
import asyncio
from util import delay

async def add_one(number: int) -> int:
    print("inside add_one()")
    return number + 1

async def hello_world_message() -> str:
    await delay(1)
    print("returning hello world")
    return 'Hello World!'

async def main() -> None:
    message = await hello_world_message()
    one_plus_one = await add_one(1)
    print(one_plus_one)
    print(message)

asyncio.run(main())


# NOTE: very imp, you'll notice that when you call delay() in hello_world_message(), both hello_world_message() AND main() are suspended... This is not ideal since we still hvae add_one() which we could have executed in this delay period of 1 second... So far, this has been pretty much like running sequential code..

# how do we achieve concurrency here? by using 'tasks.

### Distinction between regular python functions and async functions

It's fine if this goes over your head rn...

A frame is kind of a runtime context for any function, be it regular or async.. it contains local variables, instruction pointer, a reference to the bytecode of the function and a few other things...

For regular fns, this frame exists on the callstack..
For async functions however, this frame exists in the heap... More precisely, we're talking of async functions which actually USE the 'await' mechanism.. As such, the event loop can afford to work with the frame, and then just leave it alone when it needs to, and the frame will still be there when the event loop comes back for this task.

Suspended meaning that frame can just be left untouched, while the event loop pauses the task and picks other tasks... since its' on the heap, the frame survives between resumptions of the code in the async function... This allows the event loop to pick up the task where it left off...

For a regular python function, this persistence of the frame isn't possible, or needed. You called the function, that function is going to run from start to end with no pause, destroying the frame as its execution is finished, no point persisting the frame.